In [4]:
!pip install langdetect

In [5]:
import csv
import re
import pandas as pd
import nltk
import numpy as np
import joblib
from langdetect import detect, detect_langs, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from tqdm.auto import tqdm

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction import text
from sklearn.metrics import confusion_matrix

tqdm.pandas()
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [6]:
file_path = '/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/dataset.csv'
df = pd.read_csv(file_path)

In [7]:
df.head()

,label,text
0,1,Congratulations! You've been selected for a lu...
1,1,URGENT: Your account has been compromised. Cli...
2,1,You've won a free iPhone! Claim your prize by ...
3,1,Act now and receive a 50% discount on all purc...
4,1,Important notice: Your subscription will expir...


In [9]:
#Language detection
DetectorFactory.seed = 0

def lang_detect(text):
  if not isinstance(text,str) or not text.strip():
    return 'Unknown'
  try:
    return detect(text)
  except LangDetectException:
    return 'Unknown'

In [10]:
#detect language
df['language'] = df['text'].progress_apply(lang_detect)
#filter for english
df = df[df['language'] == 'en'].copy()

  0%|          | 0/45155 [00:00<?, ?it/s]

In [12]:
#regex Cleaning
def regex_clean(text):
    if not isinstance(text, str): # Ensure text is a string
        return ''

    # 1. Lowercase the text
    text = text.lower()

    # 2. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # 3. Remove URLs/Hyperlinks
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 4. Remove Email Addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # 5. Remove Numbers (often randomized in spam)
    text = re.sub(r'\d+', ' ', text)

    # 6. Remove Punctuation and Special Characters
    text = re.sub(r'[^\w\s]', ' ', text)

    # 7. Collapse multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the cleaning function to the 'text' column to create 'cleaned_text'
df['cleaned_text'] = df['text'].progress_apply(regex_clean)

  0%|          | 0/41863 [00:00<?, ?it/s]

In [14]:
#Additional cleaning - drop duplicates
df.drop_duplicates(subset=['cleaned_text'], keep='first')
df.rename(columns={'text': 'raw_text'},inplace=True)

In [16]:
lemmatizer = WordNetLemmatizer()
#POS tagging
def get_wordnet_pos_from_tag(tag):
    if not tag:
        return wordnet.NOUN
    first_letter = tag[0].upper()
    tag_dict = {
        "J": wordnet.ADJ,
        "N": wordnet.NOUN,
        "V": wordnet.VERB,
        "R": wordnet.ADV
    }
    return tag_dict.get(first_letter, wordnet.NOUN)

#Lemmatization
def lemmatize_text(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    word_tags = nltk.pos_tag(words) # Tags the whole sentence once

    lemmatized_words = [
        lemmatizer.lemmatize(word, get_wordnet_pos_from_tag(tag))
        for word, tag in word_tags
    ]
    return " ".join(lemmatized_words)

In [17]:
df['cleaned_lemmatized'] = df['cleaned_text'].progress_apply(lemmatize_text)

  0%|          | 0/41863 [00:00<?, ?it/s]

In [18]:
#exporting cleaned dataset for ease of use
#df.to_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/cleaned_dataset.csv', columns=['cleaned_lemmatized', 'label'] , index=False)

In [ ]:
#import validation dataset
df_validation = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/validation_dataset.csv')

In [22]:
#applying same nlp pipeline for validation dataset
df_validation.rename(columns={'Email Text': 'raw_text', 'Email Type': 'label'},inplace=True)
df_validation['label'] = df_validation['label'].map({'Phishing Email': 1, 'Safe Email': 0})
df_validation['cleaned_text'] = df_validation['raw_text'].progress_apply(regex_clean)
df_validation['cleaned_lemmatized'] = df_validation['cleaned_text'].progress_apply(lemmatize_text)

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

In [27]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---------------------------------------------------
# STEP 1: Split data
# TWEAK: use stratify so train/test keep the same label ratio
# ---------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_lemmatized'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# ---------------------------------------------------
# STEP 2: Build TF-IDF features
# TWEAK: add bigrams, sublinear tf, and a larger feature space
# ---------------------------------------------------
custom_stop_words = list(text.ENGLISH_STOP_WORDS) + ['escapenumber', 'escapelong', 'escapenumber escapenumber']

vectorizer = TfidfVectorizer(
    stop_words=custom_stop_words,
    max_features=10000,   # was 5000
    ngram_range=(1, 2),   # unigrams + bigrams
    sublinear_tf=True,    # reduces effect of repeated words
    min_df=2,             # ignore very rare terms
    max_df=0.95           # ignore overly common terms
)

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# ---------------------------------------------------
# STEP 3: Tune Logistic Regression
# TWEAK: give class 1 a bit more importance to catch more spam
# ---------------------------------------------------
base_model = LogisticRegression(
    max_iter=3000,
    class_weight={0: 1.0, 1: 1.25},   # increase to 1.5 if spam recall is still low
    solver='liblinear'
)

# ---------------------------------------------------
# STEP 4: Optional C tuning with CV
# TWEAK: find a better regularization strength
# Uncomment this block if you want automatic tuning.
# ---------------------------------------------------
param_grid = {
    "C": [0.1, 0.5, 1, 2, 5]
}

grid = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring="f1",   # focus on F1 instead of accuracy
    cv=5,
    n_jobs=-1
)

grid.fit(X_train_vectorized, y_train)
model = grid.best_estimator_

print("Best C:", grid.best_params_["C"])

# ---------------------------------------------------
# STEP 5: Predict using a lower threshold
# TWEAK: lower threshold from 0.50 to catch more spam
# ---------------------------------------------------
threshold = 0.40  # try 0.35 if you want even higher recall for class 1

test_probabilities = model.predict_proba(X_test_vectorized)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print("\n--- Logistic Regression Accuracy ---")
print(f"{accuracy_score(y_test, predictions) * 100:.2f}%")

print("\n--- Detailed Performance Report ---")
print(classification_report(y_test, predictions))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, predictions))

# ---------------------------------------------------
# STEP 6: Top spam words
# ---------------------------------------------------
words = vectorizer.get_feature_names_out()
spam_word_weights = model.coef_[0]
top_10_indices = np.argsort(spam_word_weights)[-10:]
top_10_words = [words[i] for i in top_10_indices]

print("\n--- Logistic Regression Top 10 Spam Words ---")
print(top_10_words[::-1])

# ---------------------------------------------------
# STEP 7: If you have a separate validation set
# Use the SAME vectorizer + model + threshold on it
# ---------------------------------------------------
validation_text = df_validation['cleaned_lemmatized']
y_val = df_validation['label']
X_val_vectorized = vectorizer.transform(validation_text)
val_probabilities = model.predict_proba(X_val_vectorized)[:, 1]
val_predictions = (val_probabilities >= threshold).astype(int)
print("\n--- Validation Accuracy ---")
print(f"{accuracy_score(y_val, val_predictions) * 100:.2f}%")
print(classification_report(y_val, val_predictions))
print(confusion_matrix(y_val, val_predictions))

Best C: 5

--- Logistic Regression Accuracy ---
97.48%

--- Detailed Performance Report ---
              precision    recall  f1-score   support

           0       0.97      0.98      0.97      4191
           1       0.98      0.97      0.97      4182

    accuracy                           0.97      8373
   macro avg       0.97      0.97      0.97      8373
weighted avg       0.97      0.97      0.97      8373


--- Confusion Matrix ---
[[4094   97]
 [ 114 4068]]

--- Logistic Regression Top 10 Spam Words ---
['http', 'info', 'claim', 'uk', 'hk', 'mobile', 'viagra', 'woman', 'health', 'symbol']

--- Validation Accuracy ---
88.15%
              precision    recall  f1-score   support

           0       0.82      0.97      0.89      1000
           1       0.97      0.79      0.87      1000

    accuracy                           0.88      2000
   macro avg       0.89      0.88      0.88      2000
weighted avg       0.89      0.88      0.88      2000

[[972  28]
 [209 791]]


In [28]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

for threshold in thresholds:

    val_probabilities = model.predict_proba(X_val_vectorized)[:, 1]

    val_predictions = (
        val_probabilities >= threshold
    ).astype(int)

    print(f"\n===== THRESHOLD: {threshold} =====")
    print(classification_report(y_val, val_predictions))
    print(confusion_matrix(y_val, val_predictions))


===== THRESHOLD: 0.3 =====
              precision    recall  f1-score   support

           0       0.90      0.91      0.91      1000
           1       0.91      0.90      0.90      1000

    accuracy                           0.91      2000
   macro avg       0.91      0.91      0.91      2000
weighted avg       0.91      0.91      0.91      2000

[[914  86]
 [103 897]]

===== THRESHOLD: 0.35 =====
              precision    recall  f1-score   support

           0       0.82      0.94      0.87      1000
           1       0.93      0.79      0.85      1000

    accuracy                           0.87      2000
   macro avg       0.87      0.87      0.86      2000
weighted avg       0.87      0.87      0.86      2000

[[940  60]
 [209 791]]

===== THRESHOLD: 0.4 =====
              precision    recall  f1-score   support

           0       0.82      0.97      0.89      1000
           1       0.97      0.79      0.87      1000

    accuracy                           0.88      20

In [31]:
#export model and vectorizer
joblib.dump(model, '/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/Models/logistic_regression_model.pkl')
joblib.dump(vectorizer, '/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/Models/tfidf_vectorizer.pkl')

['/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/Models/tfidf_vectorizer.pkl']